In [9]:
# -*- coding: utf-8 -*-
"""
按编号回填历史录取字段（openpyxl 速度优化版 V5）

核心功能：
1. 支持单个 Excel 文件，或 A/B 两个目录按省份/文件名自动配对批量处理。
2. A 文件 record-2024 == matched 或 1-1：
   用 A 文件【编号-b】匹配 B 文件【编号】，把 B 文件历史录取字段回填到 A 文件同名字段。
3. A 文件 record_direct == matched 或 1-1：
   用 A 文件【eid_direct】匹配 B 文件【编号】，把 B 文件历史录取字段回填到 A 文件同名字段。
   direct 逻辑在 record-2024 后执行，所以 direct 命中时会覆盖前面回填结果。
4. A 文件满足以下任一条件时，【是否新增】填 1：
   - indirect_key 非空
   - record_direct == 1-1
   - record-2024 == unique
5. 不生成报告文件，只输出处理后的 Excel。

速度优化：
1. 新增列不再逐行复制样式，避免 20 多个字段 * 几万行 的超慢样式复制。
2. B 文件用 read_only + iter_rows(values_only=True) 一次性构建编号索引。
3. A 文件处理时用行对象缓存，减少 ws.cell() 高频调用。
4. Jupyter 兼容：argparse 使用 allow_abbrev=False + parse_known_args()，避免 --f 被误识别。

依赖：
pip install openpyxl
"""

import os
import re
import argparse
from collections import defaultdict
from typing import Dict, List, Tuple, Optional
from openpyxl import load_workbook


# ============================================================
# 1. 路径配置：直接运行时改这里，也可以命令行参数覆盖
# ============================================================

A_INPUT_PATH = r"/Users/bangongyi/Desktop/26年招生计划清洗/Match/甘肃/Test/D"
B_INPUT_PATH = r"/Users/bangongyi/Desktop/26年招生计划清洗/Match/甘肃/mirning_test/B"
OUTPUT_DIR = r"/Users/bangongyi/Desktop/26年招生计划清洗/Match/甘肃"

OUTPUT_SUFFIX = "_已回填25录取数据"



# ============================================================
# 2. 表头配置
# ============================================================

# 如果你的表头不是第一行，改这里，例如第三行表头就改成 3
HEADER_ROW = 1

# A 文件字段
A_COL_RECORD_2024 = "record-2024"
A_COL_ID_B = "编号-b"
A_COL_RECORD_DIRECT = "record_direct"
A_COL_EID_DIRECT = "eid_direct"
A_COL_INDIRECT_KEY = "indirect_key"
A_COL_IS_NEW = "是否新增"

# B 文件字段
B_COL_ID = "编号"

# 是否只填充 A 文件空值：
# False = 直接覆盖 A 文件原值
# True  = 只有 A 文件目标字段为空时才回填
FILL_EMPTY_ONLY = False

# 是否复制新增列样式：
# False = 速度最快，推荐大文件使用
# True  = 简单复制表头/列宽，不逐行复制样式
COPY_NEW_COL_HEADER_STYLE = False

# B 文件编号重复时是否报错：
# False = 不报错，默认使用第一次出现的编号记录
# True  = 发现重复编号直接报错
RAISE_ON_DUPLICATE_B_ID = False

# A/B 目录按文件名配对时，如果同一省份识别到多个文件是否报错
RAISE_ON_DUPLICATE_PROVINCE_FILE = False

# 需要从 B 文件回填到 A 文件的字段，同名回填
RETURN_FIELDS = [
    "最近一年录取人数",
    "最近一年最低分",
    "最近一年最低位次",
    "最近二年录取人数",
    "最近二年最低分",
    "最近二年最低位次",
    "最近三年录取人数",
    "最近三年最低分",
    "最近三年最低位次",
    "最近一年最高分",
    "最近一年最高位次",
    "最近二年最高分",
    "最近二年最高位次",
    "最近三年最高分",
    "最近三年最高位次",
    "最近一年征集分",
    "最近二年征集分",
    "最近三年征集分",
    "组最近一年征集分",
    "组最近二年征集分",
    "组最近三年征集分",
]

PROVINCE_NAMES = [
    "北京", "天津", "河北", "山西", "内蒙古", "辽宁", "吉林", "黑龙江",
    "上海", "江苏", "浙江", "安徽", "福建", "江西", "山东", "河南",
    "湖北", "湖南", "广东", "广西", "海南", "重庆", "四川", "贵州",
    "云南", "西藏", "陕西", "甘肃", "青海", "宁夏", "新疆",
]

PROVINCE_PINYIN_MAP = {
    "beijing": "北京",
    "tianjin": "天津",
    "hebei": "河北",
    "shanxi": "山西",
    "neimenggu": "内蒙古",
    "liaoning": "辽宁",
    "jilin": "吉林",
    "heilongjiang": "黑龙江",
    "shanghai": "上海",
    "jiangsu": "江苏",
    "zhejiang": "浙江",
    "anhui": "安徽",
    "fujian": "福建",
    "jiangxi": "江西",
    "shandong": "山东",
    "henan": "河南",
    "hubei": "湖北",
    "hunan": "湖南",
    "guangdong": "广东",
    "guangxi": "广西",
    "hainan": "海南",
    "chongqing": "重庆",
    "sichuan": "四川",
    "guizhou": "贵州",
    "yunnan": "云南",
    "xizang": "西藏",
    "tibet": "西藏",
    "shaanxi": "陕西",
    "sanxi": "陕西",
    "gansu": "甘肃",
    "qinghai": "青海",
    "ningxia": "宁夏",
    "xinjiang": "新疆",
}


# ============================================================
# 3. 基础工具
# ============================================================

def ensure_dir(path: str) -> None:
    if path and not os.path.exists(path):
        os.makedirs(path, exist_ok=True)


def get_str(value) -> str:
    if value is None:
        return ""
    if isinstance(value, bool):
        return "TRUE" if value else "FALSE"
    if isinstance(value, int):
        return str(value)
    if isinstance(value, float):
        if value.is_integer():
            return str(int(value))
        return str(value).strip()
    return str(value).strip()


def normalize_status(value) -> str:
    """
    record 字段专用归一：
    去除空白并统一小写，便于判断 matched / unique / 1-1。
    """
    return get_str(value).replace(" ", "").replace("\u3000", "").strip().lower()


def norm_key(value) -> str:
    """
    编号匹配专用：
    - 123.0 -> 123
    - 去除常见不可见空白
    """
    s = get_str(value)
    s = s.replace("\u200b", "").replace("\xa0", "").replace("\u3000", "").strip()
    return s


def is_blank(value) -> bool:
    return get_str(value) == ""


def is_matched(value) -> bool:
    return normalize_status(value) == "matched"


def is_unique(value) -> bool:
    return normalize_status(value) == "unique"


def is_one_to_one(value) -> bool:
    return normalize_status(value) == "1-1"


def is_backfill_status(value) -> bool:
    """
    回填触发状态：
    - matched
    - 1-1

    用于：
    1. record-2024 通过【编号-b】匹配 B【编号】回填
    2. record_direct 通过【eid_direct】匹配 B【编号】回填
    """
    return is_matched(value) or is_one_to_one(value)


def safe_filename(name: str) -> str:
    s = re.sub(r'[\\/:*?"<>|]+', "_", get_str(name))
    return s or "未命名文件"


def detect_province_from_filename(path: str) -> str:
    base = os.path.basename(path)
    lower = base.lower()

    for prov in sorted(PROVINCE_NAMES, key=len, reverse=True):
        if prov in base:
            return prov

    for py, prov in sorted(PROVINCE_PINYIN_MAP.items(), key=lambda x: len(x[0]), reverse=True):
        if re.search(rf"(^|[^a-zA-Z]){re.escape(py)}([^a-zA-Z]|$)", lower):
            return prov
        if py in lower:
            return prov

    return ""


def fallback_pair_key(path: str) -> str:
    """
    省份识别失败时，用文件名主体兜底配对。
    """
    stem = os.path.splitext(os.path.basename(path))[0]
    stem = re.sub(r"[_\-\s]+", "", stem)
    stem = stem.replace("回填历史录取数据", "")
    return stem


def pair_key_from_filename(path: str) -> str:
    return detect_province_from_filename(path) or fallback_pair_key(path)


def list_excel_files(path: str) -> List[str]:
    if os.path.isfile(path):
        if os.path.basename(path).startswith("~$"):
            return []
        if path.lower().endswith((".xlsx", ".xlsm")):
            return [path]
        raise ValueError(f"输入文件不是 .xlsx/.xlsm：{path}")

    if not os.path.isdir(path):
        raise FileNotFoundError(f"路径不存在：{path}")

    files = []
    for root, _, names in os.walk(path):
        for name in names:
            if name.startswith("~$"):
                continue
            if name.lower().endswith((".xlsx", ".xlsm")):
                files.append(os.path.join(root, name))
    return sorted(files)


def build_file_map_by_name(files: List[str], label: str) -> Dict[str, str]:
    result = {}
    dup = defaultdict(list)

    for fp in files:
        key = pair_key_from_filename(fp)
        if key in result:
            dup[key].append(fp)
            if RAISE_ON_DUPLICATE_PROVINCE_FILE:
                raise ValueError(f"{label} 目录中识别到重复省份/文件键：{key}\n已存在：{result[key]}\n重复：{fp}")
            continue
        result[key] = fp

    if dup:
        print(f"[WARN] {label} 目录存在重复省份/文件键，默认使用第一次出现的文件：")
        for key, fps in dup.items():
            print(f"  - {key}：额外重复 {len(fps)} 个")

    return result


# ============================================================
# 4. Excel 表头与列工具
# ============================================================

def get_header_map(ws, header_row: int = HEADER_ROW) -> Dict[str, int]:
    header_map = {}
    for col in range(1, ws.max_column + 1):
        name = get_str(ws.cell(row=header_row, column=col).value)
        if name:
            header_map[name] = col
    return header_map


def find_col(header_map: Dict[str, int], col_name: str, required: bool = False) -> Optional[int]:
    col = header_map.get(col_name)
    if col is None and required:
        raise ValueError(f"缺少必要字段：{col_name}")
    return col


def ensure_col_fast(ws, col_name: str, header_map: Dict[str, int], header_row: int = HEADER_ROW) -> Tuple[int, Dict[str, int]]:
    """
    如果列不存在，则追加到最后。

    速度优化点：
    - 不再逐行复制样式。之前慢的主要原因就是新增 20 多列时，每列都复制整张表所有行的样式。
    - 如需保留一点格式，只复制表头样式和列宽，不复制全列单元格样式。
    """
    if col_name in header_map:
        return header_map[col_name], header_map

    new_col = ws.max_column + 1
    prev_col = max(1, new_col - 1)

    ws.cell(row=header_row, column=new_col).value = col_name

    if COPY_NEW_COL_HEADER_STYLE:
        src = ws.cell(row=header_row, column=prev_col)
        dst = ws.cell(row=header_row, column=new_col)
        if src.has_style:
            from copy import copy
            dst.font = copy(src.font)
            dst.fill = copy(src.fill)
            dst.border = copy(src.border)
            dst.alignment = copy(src.alignment)
            dst.number_format = src.number_format
            dst.protection = copy(src.protection)
        try:
            ws.column_dimensions[dst.column_letter].width = ws.column_dimensions[src.column_letter].width
        except Exception:
            pass

    header_map[col_name] = new_col
    return new_col, header_map


# ============================================================
# 5. B 文件索引构建
# ============================================================

def build_b_index(b_filepath: str) -> Tuple[Dict[str, Dict[str, object]], List[str], int]:
    """
    返回：
    - B 编号 -> 字段值 dict
    - B 文件缺失的回填字段
    - 重复编号数量
    """
    wb = load_workbook(b_filepath, read_only=True, data_only=True)
    ws = wb.worksheets[0]

    # read_only 模式下也可以读取表头行；后续用 iter_rows 一次遍历，避免 ws.cell(row, col) 反复访问。
    header_values = next(ws.iter_rows(min_row=HEADER_ROW, max_row=HEADER_ROW, values_only=True))
    header_map = {}
    for idx, value in enumerate(header_values, start=1):
        name = get_str(value)
        if name:
            header_map[name] = idx

    col_id = find_col(header_map, B_COL_ID, required=True)

    b_field_cols = {}
    missing_return_fields = []
    for field in RETURN_FIELDS:
        col = header_map.get(field)
        if col is None:
            missing_return_fields.append(field)
        else:
            b_field_cols[field] = col

    index = {}
    duplicate_count = 0

    # 转为 0-based 下标，加快取值。
    id_idx = col_id - 1
    b_field_idx = {field: col - 1 for field, col in b_field_cols.items()}

    for row_values in ws.iter_rows(min_row=HEADER_ROW + 1, values_only=True):
        if id_idx >= len(row_values):
            continue

        bid = norm_key(row_values[id_idx])
        if not bid:
            continue

        if bid in index:
            duplicate_count += 1
            if RAISE_ON_DUPLICATE_B_ID:
                raise ValueError(f"B 文件编号重复：{bid}，文件：{b_filepath}")
            continue

        values = {}
        for field, idx in b_field_idx.items():
            values[field] = row_values[idx] if idx < len(row_values) else None

        index[bid] = values

    wb.close()
    return index, missing_return_fields, duplicate_count


# ============================================================
# 6. A 文件处理
# ============================================================

def fill_return_fields_fast(row_cells, a_field_cols: Dict[str, int], b_values: Dict[str, object]) -> int:
    """
    把 B 的历史字段回填到 A 同名字段，返回实际写入字段数。
    row_cells 是 openpyxl 的当前行 cell tuple，避免 ws.cell 高频访问。
    """
    written = 0
    for field in RETURN_FIELDS:
        if field not in b_values:
            continue

        col = a_field_cols[field]
        cell = row_cells[col - 1]

        if FILL_EMPTY_ONLY and not is_blank(cell.value):
            continue

        cell.value = b_values.get(field)
        written += 1

    return written


def process_one_pair(a_filepath: str, b_filepath: str, output_dir: str) -> Dict[str, object]:
    keep_vba = a_filepath.lower().endswith(".xlsm")

    print(f"[PAIR] A={os.path.basename(a_filepath)} | B={os.path.basename(b_filepath)}")

    b_index, missing_b_fields, duplicate_b_count = build_b_index(b_filepath)
    if missing_b_fields:
        print(f"  [WARN] B 文件缺少以下回填字段，将跳过这些字段：{missing_b_fields}")
    if duplicate_b_count:
        print(f"  [WARN] B 文件存在重复编号 {duplicate_b_count} 条，默认使用第一次出现的编号记录")

    wb = load_workbook(a_filepath, keep_vba=keep_vba)
    ws = wb.worksheets[0]
    header_map = get_header_map(ws, HEADER_ROW)

    # A 文件匹配字段：允许某组字段缺失，缺失则跳过该逻辑
    col_record_2024 = find_col(header_map, A_COL_RECORD_2024, required=False)
    col_id_b = find_col(header_map, A_COL_ID_B, required=False)
    col_record_direct = find_col(header_map, A_COL_RECORD_DIRECT, required=False)
    col_eid_direct = find_col(header_map, A_COL_EID_DIRECT, required=False)
    col_indirect_key = find_col(header_map, A_COL_INDIRECT_KEY, required=False)

    if col_record_2024 is None or col_id_b is None:
        print(f"  [WARN] A 文件缺少 {A_COL_RECORD_2024} 或 {A_COL_ID_B}，跳过 record-2024 回填逻辑")
    if col_record_direct is None or col_eid_direct is None:
        print(f"  [WARN] A 文件缺少 {A_COL_RECORD_DIRECT} 或 {A_COL_EID_DIRECT}，跳过 record_direct 回填逻辑")
    if col_indirect_key is None:
        print(f"  [WARN] A 文件缺少 {A_COL_INDIRECT_KEY}，跳过 indirect_key 标记是否新增逻辑")

    # 确保 A 文件有目标字段与是否新增字段
    # 速度优化：只追加表头，不做整列样式复制。
    a_field_cols = {}
    for field in RETURN_FIELDS:
        col, header_map = ensure_col_fast(ws, field, header_map, HEADER_ROW)
        a_field_cols[field] = col

    col_is_new, header_map = ensure_col_fast(ws, A_COL_IS_NEW, header_map, HEADER_ROW)

    rows_total = max(0, ws.max_row - HEADER_ROW)
    fill_by_record_2024_rows = 0
    fill_by_direct_rows = 0
    miss_by_record_2024 = 0
    miss_by_direct = 0
    set_new_rows = 0

    # 当前 ws.max_column 已包含新增列，iter_rows 会拿到完整行。
    for row_cells in ws.iter_rows(min_row=HEADER_ROW + 1, max_row=ws.max_row, max_col=ws.max_column):
        # 读取 A 当前行必要值
        record_2024_value = row_cells[col_record_2024 - 1].value if col_record_2024 else ""
        id_b_value = row_cells[col_id_b - 1].value if col_id_b else ""

        record_direct_value = row_cells[col_record_direct - 1].value if col_record_direct else ""
        eid_direct_value = row_cells[col_eid_direct - 1].value if col_eid_direct else ""

        indirect_value = row_cells[col_indirect_key - 1].value if col_indirect_key else ""

        # 是否新增逻辑：
        # 1) indirect_key 非空
        # 2) record_direct == 1-1
        # 3) record-2024 == unique
        if (not is_blank(indirect_value)) or is_one_to_one(record_direct_value) or is_unique(record_2024_value):
            row_cells[col_is_new - 1].value = 1
            set_new_rows += 1

        # record-2024 == matched 或 1-1，用 编号-b 匹配 B 编号回填
        if col_record_2024 is not None and col_id_b is not None and is_backfill_status(record_2024_value):
            bid = norm_key(id_b_value)
            if bid and bid in b_index:
                written = fill_return_fields_fast(row_cells, a_field_cols, b_index[bid])
                if written:
                    fill_by_record_2024_rows += 1
            else:
                miss_by_record_2024 += 1

        # record_direct == matched 或 1-1，用 eid_direct 匹配 B 编号回填
        # 放在 record-2024 后执行，因此 direct 命中时会覆盖前面的同名字段。
        if col_record_direct is not None and col_eid_direct is not None and is_backfill_status(record_direct_value):
            eid = norm_key(eid_direct_value)
            if eid and eid in b_index:
                written = fill_return_fields_fast(row_cells, a_field_cols, b_index[eid])
                if written:
                    fill_by_direct_rows += 1
            else:
                miss_by_direct += 1

    ensure_dir(output_dir)
    base, ext = os.path.splitext(os.path.basename(a_filepath))
    out_path = os.path.join(output_dir, f"{safe_filename(base)}{OUTPUT_SUFFIX}{ext}")

    wb.save(out_path)
    wb.close()

    print(
        f"  [OK] 输出：{os.path.basename(out_path)} | "
        f"A数据行：{rows_total} | "
        f"record-2024回填行(matched/1-1)：{fill_by_record_2024_rows} | "
        f"direct回填行(matched/1-1)：{fill_by_direct_rows} | "
        f"是否新增=1行：{set_new_rows} | "
        f"record-2024未匹配B编号：{miss_by_record_2024} | "
        f"direct未匹配B编号：{miss_by_direct}"
    )

    return {
        "a_file": a_filepath,
        "b_file": b_filepath,
        "out_path": out_path,
        "rows_total": rows_total,
        "fill_by_record_2024_rows": fill_by_record_2024_rows,
        "fill_by_direct_rows": fill_by_direct_rows,
        "set_new_rows": set_new_rows,
        "miss_by_record_2024": miss_by_record_2024,
        "miss_by_direct": miss_by_direct,
    }


# ============================================================
# 7. 主流程
# ============================================================

def run(a_input_path: str = A_INPUT_PATH, b_input_path: str = B_INPUT_PATH, output_dir: str = OUTPUT_DIR):
    ensure_dir(output_dir)

    a_files = list_excel_files(a_input_path)
    b_files = list_excel_files(b_input_path)

    if not a_files:
        print(f"[WARN] A 路径没有发现 Excel 文件：{a_input_path}")
        return []
    if not b_files:
        print(f"[WARN] B 路径没有发现 Excel 文件：{b_input_path}")
        return []

    a_is_single = os.path.isfile(a_input_path)
    b_is_single = os.path.isfile(b_input_path)

    results = []

    print("=" * 100)
    print("开始按编号回填历史录取字段（速度优化版）")
    print(f"A 输入：{a_input_path}")
    print(f"B 输入：{b_input_path}")
    print(f"输出目录：{output_dir}")
    print("=" * 100)

    # 单文件对单文件
    if a_is_single and b_is_single:
        results.append(process_one_pair(a_files[0], b_files[0], output_dir))
        print("=" * 100)
        print("处理完成")
        return results

    # 目录或混合模式：按省份/文件名键配对
    a_map = build_file_map_by_name(a_files, "A")
    b_map = build_file_map_by_name(b_files, "B")

    common_keys = sorted(set(a_map.keys()) & set(b_map.keys()))
    only_a = sorted(set(a_map.keys()) - set(b_map.keys()))
    only_b = sorted(set(b_map.keys()) - set(a_map.keys()))

    if only_a:
        print(f"[WARN] 以下 A 文件未找到对应 B 文件，跳过：{only_a}")
    if only_b:
        print(f"[WARN] 以下 B 文件未找到对应 A 文件，跳过：{only_b}")

    if not common_keys:
        print("[WARN] 没有找到可配对处理的 A/B 文件")
        return []

    for key in common_keys:
        try:
            results.append(process_one_pair(a_map[key], b_map[key], output_dir))
        except Exception as e:
            print(f"[FAIL] 配对键：{key} | A={a_map.get(key)} | B={b_map.get(key)} | {e}")

    print("=" * 100)
    print(f"处理完成：成功输出 {len(results)} 个文件")
    print("=" * 100)
    return results


# allow_abbrev=False 用于避免 Jupyter 自动传入的 --f=kernel.json 被 argparse 误识别为 --fill-empty-only。
def parse_args():
    parser = argparse.ArgumentParser(
        description="根据 A 文件匹配字段，从 B 文件按编号回填历史录取字段（速度优化版）",
        allow_abbrev=False,
    )
    parser.add_argument("--a-input", default=A_INPUT_PATH, help="A 文件或 A 文件目录")
    parser.add_argument("--b-input", default=B_INPUT_PATH, help="B 文件或 B 文件目录")
    parser.add_argument("--output-dir", default=OUTPUT_DIR, help="输出目录")
    parser.add_argument("--header-row", type=int, default=HEADER_ROW, help="表头行号，默认 1")
    parser.add_argument("--fill-empty-only", action="store_true", help="只在 A 目标字段为空时回填，不覆盖已有值")
    parser.add_argument("--copy-header-style", action="store_true", help="新增列时只复制表头样式和列宽；默认不复制，速度最快")
    args, _unknown = parser.parse_known_args()
    return args


def main():
    global HEADER_ROW, FILL_EMPTY_ONLY, COPY_NEW_COL_HEADER_STYLE

    args = parse_args()
    HEADER_ROW = args.header_row

    if args.fill_empty_only:
        FILL_EMPTY_ONLY = True

    if args.copy_header_style:
        COPY_NEW_COL_HEADER_STYLE = True

    run(
        a_input_path=args.a_input,
        b_input_path=args.b_input,
        output_dir=args.output_dir,
    )


if __name__ == "__main__":
    main()


开始按编号回填历史录取字段（速度优化版）
A 输入：/Users/bangongyi/Desktop/26年招生计划清洗/Match/甘肃/Test/D
B 输入：/Users/bangongyi/Desktop/26年招生计划清洗/Match/甘肃/mirning_test/B
输出目录：/Users/bangongyi/Desktop/26年招生计划清洗/Match/甘肃
[PAIR] A=gansu_26_0617-2_eric_result_a-final.xlsx | B=已补代码_甘肃6.4_1_fine_new_eric.xlsx
  [OK] 输出：gansu_26_0617-2_eric_result_a-final_已回填25录取数据.xlsx | A数据行：32050 | record-2024回填行(matched/1-1)：21846 | direct回填行(matched/1-1)：1019 | 是否新增=1行：7986 | record-2024未匹配B编号：0 | direct未匹配B编号：0
处理完成：成功输出 1 个文件
